# Day 30: Scaled Dot-Product Attention with Masking

**Layer:** Implementation | **Prerequisite:** Day 02 (Transformer), Day 29 (Decoder)


## Concept Overview

Scaled dot-product attention (SDPA) is the core computation of transformers. For queries Q, keys K, and values V: Attention(Q,K,V) = softmax(QKᵀ/√d_head) · V. The causal mask ensures tokens only attend to earlier positions. Understanding SDPA in detail reveals why FlashAttention and GQA are so important.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. SDPA Implementation


In [ ]:
def sdpa(Q, K, V, mask=None, dropout_p=0.0):
    """
    Scaled Dot-Product Attention.
    Q,K,V: [batch, heads, seq, d_head]
    Returns: [batch, heads, seq, d_head]
    """
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d ** 0.5)  # [B, H, T, T]
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    weights = F.dropout(weights, p=dropout_p)
    return weights @ V, weights

def causal_mask(seq_len, device='cpu'):
    """Lower triangular causal mask (attend to self + past)."""
    return torch.tril(torch.ones(seq_len, seq_len, device=device))

# Test
torch.manual_seed(42)
B, H, T, d = 2, 4, 16, 32
Q = torch.randn(B, H, T, d)
K = torch.randn(B, H, T, d)
V = torch.randn(B, H, T, d)
mask = causal_mask(T)

out, weights = sdpa(Q, K, V, mask)
print(f'Input shape:  Q={Q.shape}')
print(f'Output shape: {out.shape}')
print(f'Attention weights shape: {weights.shape}')
print(f'Weights sum to 1 per query: {weights[0,0].sum(dim=-1).allclose(torch.ones(T))} (unmasked)')

# Verify causal: weight[i,j]=0 for j>i
causal_ok = (weights[0,0].triu(diagonal=1).abs().max() < 1e-6).item()
print(f'Causal masking correct: {causal_ok}')


## 2. Visualizing Attention Patterns


In [ ]:
# Visualize attention weights
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Causal attention
ax = axes[0]
im = ax.imshow(weights[0, 0].detach().numpy(), cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title('Causal Attention Weights\n(Lower triangular)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

# Full (bidirectional) attention
_, weights_full = sdpa(Q, K, V)  # no mask
ax = axes[1]
im = ax.imshow(weights_full[0, 0].detach().numpy(), cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title('Bidirectional Attention Weights\n(Full)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

plt.tight_layout()
plt.savefig('attention_patterns.png', dpi=100, bbox_inches='tight')
plt.show()


## 3. Memory and Compute Scaling


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def benchmark_sdpa(seq_len, d=64, heads=8, batch=1, iters=20):
    Q = torch.randn(batch, heads, seq_len, d, device=device, dtype=torch.float16)
    K = torch.randn_like(Q); V = torch.randn_like(Q)
    mask = causal_mask(seq_len, device=device)
    # Warmup
    for _ in range(3): sdpa(Q, K, V, mask)
    if device=='cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters): sdpa(Q, K, V, mask)
    if device=='cuda': torch.cuda.synchronize()
    return (time.perf_counter()-t0)/iters*1000

def attn_memory_mb(seq_len, heads=8, batch=1):
    """Memory for attention score matrix (FP16)."""
    return batch * heads * seq_len * seq_len * 2 / 1e6

print(f'{"Seq Len":>10} {"Time (ms)":>12} {"Attn Matrix (MB)":>20}')
for seq in [64, 128, 256, 512, 1024, 2048]:
    t = benchmark_sdpa(seq)
    mem = attn_memory_mb(seq)
    print(f'{seq:>10} {t:>12.3f} {mem:>20.1f}')
print('Note: attention matrix grows O(seq²) — this is what FlashAttention eliminates from HBM')


## 4. Flash Attention vs Standard SDPA


In [ ]:
# Compare PyTorch's SDPA backends
if torch.cuda.is_available():
    for backend, use_flash in [('Standard', False), ('Flash/Mem-Eff', True)]:
        Q2 = torch.randn(2, 8, 512, 64, device='cuda', dtype=torch.float16)
        K2, V2 = torch.randn_like(Q2), torch.randn_like(Q2)
        # Warmup
        for _ in range(5):
            if use_flash:
                F.scaled_dot_product_attention(Q2, K2, V2, is_causal=True)
            else:
                sdpa(Q2, K2, V2, causal_mask(512, 'cuda'))
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(50):
            if use_flash:
                out = F.scaled_dot_product_attention(Q2, K2, V2, is_causal=True)
            else:
                out, _ = sdpa(Q2, K2, V2, causal_mask(512, 'cuda'))
        torch.cuda.synchronize()
        t_ms = (time.perf_counter()-t0)/50*1000
        print(f'{backend}: {t_ms:.3f} ms (seq=512, batch=2, h=8)')
else:
    print('Flash attention comparison requires CUDA GPU')
    print('F.scaled_dot_product_attention() automatically uses FlashAttention when available')


## Experiments

1. Measure the inflection point where standard SDPA runs OOM vs FlashAttention.
2. Implement multi-query attention (MQA) where K,V have 1 head while Q has N heads.
3. Implement ALiBi position bias instead of causal mask — how does it affect the attention pattern?


## Key Takeaways

- SDPA has O(n²) time and space complexity in sequence length — the attention score matrix is seq×seq.
- Causal masking is implemented by filling future positions with -inf before softmax, producing zero attention weight.
- F.scaled_dot_product_attention() automatically dispatches to FlashAttention when available — use it, never write a manual SDPA in production.
- The attention score matrix is the primary memory bottleneck for long-context inference — 1K seq, 32 heads, FP16 = 64MB just for scores.

## References
- Vaswani et al. (2017), "Attention Is All You Need"
- Dao et al. (2022), "FlashAttention"
